# Week 8 — The Transformer, Built End-to-End

The architecture that changed everything. We implement the full Transformer from Vaswani et al. (2017) in PyTorch — multi-head attention, positional encoding, residuals, layer norm, the works — *without* using `nn.Transformer`. We then benchmark our implementation against PyTorch's built-in one for correctness.

## Learning Objectives

- Derive scaled dot-product attention and explain the $\sqrt{d_k}$ scaling.
- Implement multi-head attention from scratch in PyTorch.
- Implement sinusoidal positional encoding and discuss RoPE and ALiBi as modern alternatives.
- Assemble and train a full encoder–decoder Transformer.

## Required Reading

- Vaswani, A., et al. (2017). *Attention Is All You Need*.
- Su, J., et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding*.
- Press, O., Smith, N. A., & Lewis, M. (2022). *Train Short, Test Long: Attention with Linear Biases*.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)

## 1. Scaled dot-product attention

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V.$$

**Why $\sqrt{d_k}$?** If $Q$ and $K$ have entries with unit variance, then $(QK^\top)_{ij}$ has variance $d_k$. Without the scaling, large $d_k$ pushes softmax into saturation, where gradients vanish. Dividing by $\sqrt{d_k}$ restores unit variance and keeps the softmax in its informative regime.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V: (..., T, d)
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# Sanity check: attention output preserves the value dim, weights sum to 1.
Q = torch.randn(2, 5, 8); K = torch.randn(2, 5, 8); V = torch.randn(2, 5, 8)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"output shape: {tuple(out.shape)};  weights sum to 1: {torch.allclose(w.sum(-1), torch.ones(2, 5))}")

## 2. Multi-head attention

Project $Q, K, V$ into $h$ subspaces of dimension $d_k = d_{\text{model}}/h$, run scaled dot-product attention in each, concatenate, project back.

$$\text{MHA}(X) = [\text{head}_1, \ldots, \text{head}_h] W_O, \quad \text{head}_i = \text{Attn}(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)}).$$

We implement this with a single batched matmul rather than a Python loop over heads.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, Tq, _ = q.shape
        Tk = k.size(1)
        Q = self.W_q(q).view(B, Tq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, Tk, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, Tk, self.n_heads, self.d_k).transpose(1, 2)
        # mask broadcast over heads: (B, 1, Tq, Tk) or (1, 1, Tq, Tk)
        if mask is not None and mask.dim() == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.d_model)
        return self.W_o(out)

mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)
print(f"MHA output: {tuple(mha(x, x, x).shape)} (should be (2, 10, 64))")

## 3. Positional encoding

Self-attention is permutation-equivariant — it has no notion of position by itself. The original Transformer adds **sinusoidal** position vectors:

$$\text{PE}_{(\text{pos}, 2i)} = \sin(\text{pos}/10000^{2i/d}), \quad \text{PE}_{(\text{pos}, 2i+1)} = \cos(\text{pos}/10000^{2i/d}).$$

The key property: any shift $\text{PE}_{\text{pos}+k}$ is a linear function of $\text{PE}_{\text{pos}}$, so the model can learn relative positions even though the encoding is absolute.

**Modern alternatives.**

- **RoPE** (Su et al., 2021): rotate $Q$ and $K$ by a position-dependent angle.
- **ALiBi** (Press et al., 2022): add a static bias to attention scores proportional to $-|i - j|$, with no positional embeddings at all. Extrapolates to longer sequences than seen at training.

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]

pe = SinusoidalPE(d_model=64, max_len=100).pe.numpy()
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe.T, aspect='auto', cmap='RdBu')
ax.set(xlabel='position', ylabel='dim', title='Sinusoidal positional encoding (64 dim, 100 positions)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 4. The full Transformer block

Pre-norm variant (more stable to train than the original post-norm):

$$\mathbf{x} \leftarrow \mathbf{x} + \text{MHA}(\text{LN}(\mathbf{x})), \qquad \mathbf{x} \leftarrow \mathbf{x} + \text{FFN}(\text{LN}(\mathbf{x})).$$

The FFN expansion ratio is conventionally 4× (i.e., hidden dim = $4 d_{\text{model}}$).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = x + self.drop(self.attn(self.ln1(x), self.ln1(x), self.ln1(x), src_mask))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, mem_mask=None):
        h = self.ln1(x)
        x = x + self.drop(self.self_attn(h, h, h, tgt_mask))
        h = self.ln2(x)
        x = x + self.drop(self.cross_attn(h, memory, memory, mem_mask))
        x = x + self.drop(self.ffn(self.ln3(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, V, d_model=64, n_heads=4, d_ff=256, n_layers=2, max_len=64, pad=0):
        super().__init__()
        self.pad = pad
        self.src_emb = nn.Embedding(V, d_model, padding_idx=pad)
        self.tgt_emb = nn.Embedding(V, d_model, padding_idx=pad)
        self.pe = SinusoidalPE(d_model, max_len)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)

    def _make_pad_mask(self, x):
        return (x == self.pad).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)

    def _make_causal_mask(self, T, device):
        return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, src, tgt):
        src_pad = self._make_pad_mask(src)
        tgt_pad = self._make_pad_mask(tgt)
        causal = self._make_causal_mask(tgt.size(1), tgt.device)
        tgt_mask = tgt_pad | causal  # broadcast

        x = self.pe(self.src_emb(src))
        for layer in self.encoder:
            x = layer(x, src_pad)
        memory = x

        y = self.pe(self.tgt_emb(tgt))
        for layer in self.decoder:
            y = layer(y, memory, tgt_mask, src_pad)
        return self.head(self.ln_out(y))

## 5. Training on the digit-reversal task

We reuse the digit-reversal task from Week 7 — but now with a Transformer.

In [ ]:
DIGITS = ['zero','one','two','three','four','five','six','seven','eight','nine']
SPECIALS = ['<pad>', '<s>', '</s>']
VOCAB = SPECIALS + DIGITS
tok2id = {t: i for i, t in enumerate(VOCAB)}
id2tok = {i: t for t, i in tok2id.items()}
V = len(VOCAB)
PAD, BOS, EOS = tok2id['<pad>'], tok2id['<s>'], tok2id['</s>']

def make_batch(N, length, max_len=12):
    src = torch.full((N, max_len), PAD)
    tgt = torch.full((N, max_len + 2), PAD)
    for i in range(N):
        digits = np.random.randint(0, 10, size=length)
        src[i, :length] = torch.tensor([tok2id[DIGITS[d]] for d in digits])
        tgt[i, 0] = BOS
        for j, d in enumerate(digits[::-1]):
            tgt[i, j + 1] = tok2id[DIGITS[d]]
        tgt[i, length + 1] = EOS
    return src, tgt

model = Transformer(V, d_model=64, n_heads=4, d_ff=256, n_layers=2, max_len=16, pad=PAD)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

for step in range(800):
    L = np.random.choice([4, 5, 6, 7])
    src, tgt = make_batch(64, L)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    logits = model(src, tgt_in)
    loss = F.cross_entropy(logits.reshape(-1, V), tgt_out.reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 200 == 0:
        print(f"step {step+1:4d}  loss = {loss.item():.3f}")

# Evaluate
@torch.no_grad()
def accuracy(L, n=200):
    src, tgt = make_batch(n, L)
    logits = model(src, tgt[:, :-1])
    preds = logits.argmax(-1)
    tgt_out = tgt[:, 1:]
    mask = (tgt_out != PAD)
    return ((preds == tgt_out) | ~mask).all(dim=1).float().mean().item()

print()
for L in [4, 6, 8, 10]:
    print(f"  length {L:2d}: accuracy = {accuracy(L):.3f}")

## 6. Causal masking visualization

The decoder self-attention must not look ahead. The mask is upper-triangular `True` — those positions get `-inf` before softmax.

In [ ]:
causal = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1).numpy()
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(~causal, cmap='Greens')
for i in range(8):
    for j in range(8):
        ax.text(j, i, '✓' if not causal[i, j] else '✗', ha='center', va='center',
                color='black' if not causal[i, j] else 'red', fontsize=12)
ax.set(title='Causal mask (✓ = allowed to attend)', xlabel='key position', ylabel='query position')
ax.set_xticks(range(8)); ax.set_yticks(range(8))
plt.tight_layout(); plt.show()

## 7. RoPE — rotary position embedding

Rather than adding a position vector to the embedding, RoPE rotates pairs of dimensions of $Q$ and $K$ by an angle that depends on position. The resulting dot product depends only on relative position $i - j$ — exactly the property we want.

For a 2D pair $(x_1, x_2)$ at position $m$:

$$R_\theta^{(m)} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}.$$

In [ ]:
def apply_rope(x, base=10000):
    # x: (..., T, d); d must be even. Rotate consecutive pairs.
    T, d = x.size(-2), x.size(-1)
    assert d % 2 == 0
    half = d // 2
    inv_freq = 1.0 / (base ** (torch.arange(0, half).float() / half))
    pos = torch.arange(T).float().unsqueeze(1)               # (T, 1)
    freqs = pos * inv_freq.unsqueeze(0)                       # (T, half)
    cos = torch.cos(freqs); sin = torch.sin(freqs)
    x1, x2 = x[..., :half], x[..., half:]
    x_rot1 = x1 * cos - x2 * sin
    x_rot2 = x1 * sin + x2 * cos
    return torch.cat([x_rot1, x_rot2], dim=-1)

# Verify relative-position property: rotating both Q and K by their positions
# leaves Q·K dependent only on (m - n).
d = 16
q = torch.randn(d); k = torch.randn(d)
for (m1, n1), (m2, n2) in [((3, 1), (7, 5)), ((0, 2), (8, 10))]:
    Q1 = apply_rope(q.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, m1]
    K1 = apply_rope(k.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, n1]
    Q2 = apply_rope(q.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, m2]
    K2 = apply_rope(k.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, n2]
    print(f"m-n = {m1-n1}: Q·K = {(Q1@K1).item():+.4f}    "
          f"m-n = {m2-n2}: Q·K = {(Q2@K2).item():+.4f}    "
          f"equal? {torch.allclose(Q1@K1, Q2@K2, atol=1e-4)}")

## 8. Exercises

1. **Why $\sqrt{d_k}$?** Empirically demonstrate: as $d_k$ grows, the variance of unscaled scores grows linearly. Without the scaling, the resulting softmax is nearly one-hot, gradients vanish, and training stalls.
2. **Pre-norm vs. post-norm.** Implement both variants. Train each with the same hyperparameters. Plot training loss curves. Pre-norm should be smoother and more forgiving of learning-rate choice (Xiong et al., 2020).
3. **RoPE for length extrapolation.** Swap sinusoidal PE for RoPE. Train on sequences up to length 8, evaluate at length 16. Compare against the sinusoidal model. Which extrapolates better?
4. **Match PyTorch.** With careful initialization, the implementation above should match `torch.nn.Transformer` output to within float precision. Verify this on a single batch.

---

## Next Week

Week 9 — Pretrained models. BERT, GPT, T5 — three philosophies of pretraining.